# Volume 8, Chapter 2: Vector Fields and Flows (Python companion)

Numerical integrators that respect the geometry of a flow, parallel parking with Lie brackets, and the bracket measured as the gap of a small loop.

*Before each solution, predict what the output will look like.*

## Exercises

**P2.1 (★) CS application: Euler versus leapfrog on the pendulum.** The pendulum $\theta''=-\sin\theta$ is the flow of the field $(\theta,\omega)\mapsto(\omega,-\sin\theta)$ on $\R^2$ (or on the cylinder). (a) Integrate it from $(\theta,\omega)=(2,0)$ up to $t=200$ with the explicit Euler method and with the leapfrog (Störmer--Verlet) method, step $h=0.05$, and plot the energy $E=\frac12\omega^2-\cos\theta$. (b) Plot both trajectories in the phase plane. (c) Measure how the global error at time $t=10$ depends on $h$ for both methods. (d) Integrate the variational equation along the exact trajectory and plot $\|D_x\varphi(t,x)\|$: how fast do initial errors grow on this system?

**P2.2 (★★) CS application: parallel parking from brackets.** (a) For the unicycle of Exercise 2.5, run the loop turn $s$, drive $s$, turn $-s$, drive $-s$ for $s=2^{-1},\dots,2^{-8}$ and check that the sideways displacement is $s^2+O(s^4)$ and the forward error $-\frac{s^3}2+O(s^5)$. (b) Park: starting at $(0,0,0)$, reach $(0,1,0)$ (one unit to the left, same heading) using only the two allowed motions, by repeating a small loop $N$ times. How does the total distance driven depend on $N$, and what is its limit? (c) A car has a steering angle $\phi$ too: $D=\cos\theta\,\partial_x+\sin\theta\,\partial_y+\tan\phi\,\partial_\theta$ (drive, wheelbase $1$) and $S=\partial_\phi$ (steer). Compute $[S,D]$ and $[[S,D],D]$ symbolically and show that $D,S,[S,D],[[S,D],D]$ span all four directions.

**P2.3 (★) The bracket, measured.** (a) For two random polynomial vector fields on $\R^2$, compute the loop $\psi(s)$ of Theorem 2.10 numerically (with an accurate ODE solver) and compare $\frac{\psi(s)-p}{s^2}$ with $[X,Y](p)$ from the formula $DY\cdot X-DX\cdot Y$. (b) Check the Jacobi identity symbolically for three random fields. (c) For the plane field of Example 2.15, follow a small loop in the allowed directions and measure how far it climbs in $z$: compare with the area enclosed by the projection of the loop to the $(x,y)$-plane.

## Solutions

### P2.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def euler(th, om, h, n):
    out = np.empty((n + 1, 2)); out[0] = th, om
    for k in range(n):
        th, om = th + h * om, om - h * np.sin(th)
        out[k + 1] = th, om
    return out

def leapfrog(th, om, h, n):
    out = np.empty((n + 1, 2)); out[0] = th, om
    for k in range(n):
        om_half = om - 0.5 * h * np.sin(th)
        th = th + h * om_half
        om = om_half - 0.5 * h * np.sin(th)
        out[k + 1] = th, om
    return out

energy = lambda z: 0.5 * z[:, 1] ** 2 - np.cos(z[:, 0])
h, T = 0.05, 200.0
n = int(T / h)
tt = np.linspace(0, T, n + 1)
E0 = 0.5 * 0 - np.cos(2.0)
zE, zL = euler(2.0, 0.0, h, n), leapfrog(2.0, 0.0, h, n)
print(f"initial energy {E0:.6f}")
print(f"Euler:    energy at t=200: {energy(zE)[-1]:.4f}")
print(f"leapfrog: max |E - E0| over [0,200]: {np.abs(energy(zL) - E0).max():.2e}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].plot(tt, energy(zE), label="Euler"); ax[0].plot(tt, energy(zL), label="leapfrog")
ax[0].axhline(1, color="gray", lw=0.6, ls=":"); ax[0].set_xlabel("t"); ax[0].set_ylabel("energy"); ax[0].legend()
ax[1].plot(zE[:, 0], zE[:, 1], lw=0.6, label="Euler"); ax[1].plot(zL[:, 0], zL[:, 1], lw=0.8, label="leapfrog")
ax[1].set_xlabel(r"$\theta$"); ax[1].set_ylabel(r"$\omega$"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# (c) order of accuracy at t = 10
field = lambda t, z: [z[1], -np.sin(z[0])]
ref = solve_ivp(field, (0, 10), [2.0, 0.0], rtol=1e-13, atol=1e-13).y[:, -1]
print("    h        Euler error    leapfrog error")
for h in (0.1, 0.05, 0.025, 0.0125):
    n = int(round(10 / h))
    eE = np.linalg.norm(euler(2.0, 0.0, h, n)[-1] - ref)
    eL = np.linalg.norm(leapfrog(2.0, 0.0, h, n)[-1] - ref)
    print(f"{h:7.4f}   {eE:12.3e}   {eL:12.3e}")

In [ ]:
# (d) the variational equation Y' = DF(phi) Y along the trajectory from (2, 0)
def aug(t, u):
    th, om = u[:2]; Y = u[2:].reshape(2, 2)
    A = np.array([[0.0, 1.0], [-np.cos(th), 0.0]])
    return np.concatenate([[om, -np.sin(th)], (A @ Y).ravel()])

sol = solve_ivp(aug, (0, 200), np.concatenate([[2.0, 0.0], np.eye(2).ravel()]),
                rtol=1e-11, atol=1e-11, dense_output=True)
ts = np.linspace(0, 200, 2001)
norms = [np.linalg.norm(sol.sol(t)[2:].reshape(2, 2), 2) for t in ts]
dets = [np.linalg.det(sol.sol(t)[2:].reshape(2, 2)) for t in ts]
plt.figure(figsize=(6, 3.5)); plt.plot(ts, norms); plt.xlabel("t"); plt.ylabel(r"$\|D_x\varphi(t,x)\|$"); plt.show()
print("norm at t = 50, 100, 200:", [round(norms[i], 2) for i in (500, 1000, 2000)])
print("det D_x phi stays", min(dets), max(dets))

(a)--(b) The pendulum conserves $E=\frac12\omega^2-\cos\theta$, and its orbits are the level curves of $E$. Euler's method gains energy at every step: the computed pendulum swings higher and higher, and well before $t=200$ its energy passes $1$, the energy of the upright position, after which it goes over the top and spins for ever. Its phase portrait is an outward spiral. The leapfrog method keeps the energy within about $10^{-3}$ of its initial value for the whole run, oscillating without drift, and its orbit is a closed curve indistinguishable from the true one.

(c) Halving $h$ halves the Euler error and divides the leapfrog error by four: they are methods of order $1$ and $2$. But the energy behaviour in (a) is not explained by the order. The leapfrog map is itself the exact time-$h$ flow of a slightly perturbed conservative system (it is *symplectic*: it preserves area in the phase plane, like the true flow), so it inherits a conserved quantity close to $E$. Euler's map multiplies areas by $1+h^2\cos\theta$, on average more than $1$ near the bottom, and the orbits spiral out. Geometry of the flow, not just accuracy, is what one wants to preserve over long times.

(d) The norm of $D_x\varphi$ grows roughly linearly in $t$: neighbouring orbits have slightly different periods, so they drift apart in phase at a constant rate, but there is no exponential separation. The determinant stays equal to $1$: by Liouville's formula, $\frac{\dd}{\dd t}\det Y=\operatorname{tr}(DF)\det Y=0$, since the field has divergence zero. That is the area preservation the leapfrog method copies.

### P2.2

In [ ]:
import numpy as np

def turn(z, s):   return z + np.array([0, 0, s])
def drive(z, s):  return z + np.array([s * np.cos(z[2]), s * np.sin(z[2]), 0])

def loop(z, s):
    return drive(turn(drive(turn(z, s), s), -s), -s)

print("    s          dy/s^2        (dx + s^3/2)/s^5")
for k in range(1, 9):
    s = 2.0 ** -k
    z = loop(np.zeros(3), s)
    print(f"{s:10.6f}   {z[1]/s**2:.10f}   {(z[0] + s**3/2)/s**5:.6f}")

In [ ]:
# (b) park one unit to the left with N loops of size s, where N s^2 is about 1
def park(N):
    z = np.zeros(3)
    s = np.sqrt(1.0 / N)
    for _ in range(3):                         # adjust s so that the total sideways shift is exactly 1
        dy = loop(np.zeros(3), s)[1]
        s = s * np.sqrt((1.0 / N) / dy)
    for _ in range(N):
        z = loop(z, s)
    return z, s, 2 * N * s                      # distance driven: two drives of length s per loop

for N in (1, 4, 16, 64, 256, 1024):
    z, s, dist = park(N)
    print(f"N={N:5d}  s={s:.4f}  final=({z[0]:+.5f}, {z[1]:.5f}, {z[2]:+.1e})  distance driven = {dist:.2f}")

In [ ]:
# (c) the car: brackets with sympy
import sympy as sp
x, y, th, ph = sp.symbols('x y theta phi')
q = sp.Matrix([x, y, th, ph])
D = sp.Matrix([sp.cos(th), sp.sin(th), sp.tan(ph), 0])
S = sp.Matrix([0, 0, 0, 1])

def br(X, Y):              # [X, Y] = DY.X - DX.Y
    return sp.simplify(Y.jacobian(q) * X - X.jacobian(q) * Y)

W = br(S, D); L = br(W, D)
print("[S,D]     =", list(W))
print("[[S,D],D] =", list(L))
M = sp.Matrix.hstack(D, S, W, L)
print("det[D, S, [S,D], [[S,D],D]] =", sp.simplify(M.det()))

(a) The sideways shift divided by $s^2$ tends to $1$ and the forward error matches $-\frac{s^3}2$ to the next order, exactly as the computation of Exercise 2.5 predicts ($s\cos s-s=-\frac{s^3}2+\frac{s^5}{24}-\dots$, so the last column tends to $\frac1{24}\approx0.0417$). The heading always returns exactly to $0$.

(b) With $N$ loops of size $s\approx N^{-1/2}$, the car moves one unit sideways, and the leftover forward error is $N\cdot O(s^3)=O(N^{-1/2})$, which also shrinks. The distance driven is $2Ns\approx2\sqrt N$, which *grows* without bound: the cost of moving in a forbidden direction is not the length of the move but the number of small manoeuvres, because each loop of size $s$ buys only $s^2$. A good driver therefore uses a few large loops, accepting the higher-order errors and correcting them, which is what the planners used in robotics do with the full nonlinear system rather than the quadratic approximation.

(c) $[S,D]=\frac1{\cos^2\phi}\partial_\theta$ (steering then driving makes the car rotate: ``rotate'') and $[[S,D],D]=\frac{-\sin\theta\,\partial_x+\cos\theta\,\partial_y}{\cos^2\phi}$ (rotating then driving makes it slide sideways: ``slide''). The determinant of the four fields is $\frac{1}{\cos^4\phi}$ up to sign, never zero: the brackets reach every direction, which is the hypothesis of Chow's theorem and the reason a car can reach every position and heading. Nelson's names for the four fields are drive, steer, rotate and slide.

### P2.3

In [ ]:
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp

rng = np.random.default_rng(3)
X1, X2 = sp.symbols('x1 x2'); q = sp.Matrix([X1, X2])

def random_field():
    return sp.Matrix([sum(int(rng.integers(-3, 4)) * X1**i * X2**j for i in range(3) for j in range(3 - i)) / 3
                      for _ in range(2)])

def br(X, Y):
    return sp.expand(Y.jacobian(q) * X - X.jacobian(q) * Y)

X, Y = random_field(), random_field()
fX = sp.lambdify((X1, X2), list(X)); fY = sp.lambdify((X1, X2), list(Y))

def flow(f, p, t):
    if t == 0: return np.array(p)
    return solve_ivp(lambda _, z: f(*z), (0, t), p, rtol=1e-13, atol=1e-14).y[:, -1]

p = np.array([0.3, -0.2])
exact = np.array([float(c.subs({X1: p[0], X2: p[1]})) for c in br(X, Y)])
print("X =", list(X)); print("Y =", list(Y))
print("[X,Y](p) from the formula:", exact)
for s in (0.1, 0.05, 0.025, 0.0125):
    z = flow(fY, flow(fX, flow(fY, flow(fX, p, s), s), -s), -s)
    est = (z - p) / s**2
    print(f"s={s:7.4f}  (psi(s)-p)/s^2 = {est}   error = {np.linalg.norm(est - exact):.2e}")

In [ ]:
# (b) the Jacobi identity
Z = random_field()
jac = br(X, br(Y, Z)) + br(Y, br(Z, X)) + br(Z, br(X, Y))
print("Jacobi:", list(sp.simplify(jac)))

In [ ]:
# (c) loops tangent to the plane field ker(dz - y dx): moving along Y1 = d/dy and Y2 = d/dx + y d/dz.
# Follow a closed curve (x(t), y(t)) in the plane and lift it: z' = y x'.
t = np.linspace(0, 2 * np.pi, 20001)
for r in (0.5, 0.2, 0.1):
    xs, ys = r * np.cos(t), r * np.sin(t)
    dz = np.trapezoid(ys * np.gradient(xs, t), t)
    print(f"circle of radius {r}: climb in z = {dz:+.6f},  -area = {-np.pi * r**2:+.6f}")
sq = 0.3                                     # the square loop of Theorem 2.10 with X = Y2, Y = Y1
z = 0.0; y = 0.0
z += y * sq; y += sq; z += y * (-sq); y -= sq
print(f"square loop of side {sq}: climb {z:+.4f} = -s^2 = {-sq**2:+.4f}")

(a) The finite-difference estimate converges to the formula $DY\cdot X-DX\cdot Y$, with error proportional to $s$: the $O(s^3)$ term of Theorem 2.10 divided by $s^2$. This is the bracket measured as a physical displacement, with no derivatives taken.

(b) The Jacobi identity holds exactly, as Proposition 2.9 says it must.

(c) Lifting a closed curve of the plane to the plane field means solving $z'=yx'$, so the climb after one turn is $\oint y\dd x$, which is minus the enclosed area (Green's formula, Chapter 3). The climb depends only on the area of the loop, not on its shape, and is quadratic in its size: the bracket $[Y_1,Y_2]=\partial_z$ multiplies area. For the square loop of Theorem 2.10 (first along $Y_2=\partial_x+y\partial_z$, then $Y_1=\partial_y$, then back) the climb is $-s^2$, in agreement with $s^2[Y_2,Y_1]=-s^2\partial_z$. This is the *holonomy* of the plane field, and Chapter 4 will meet the same phenomenon for parallel transport on a curved surface, where the rotation after a loop is the enclosed curvature.